In [ ]:
# @title
!pip install torch torchvision torchaudio tqdm scikit-image pillow matplotlib opencv-python


In [ ]:
!mkdir -p /content/images/train /content/images/val

# Example: download COCO 2017 val set (1 GB+)
!wget -q http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
!unzip -q /content/val2017.zip -d /content/images/val
!rm /content/val2017.zip   # cleanup zip to save space

# (Optional) download COCO 2017 test set too
!wget -q http://images.cocodataset.org/zips/test2017.zip -O /content/test2017.zip
!unzip -q /content/test2017.zip -d /content/images/train
!rm /content/test2017.zip


In [ ]:
# eecv16 with pretrained weights

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
import numpy as np
import torch
import cv2
from pathlib import Path

# --- your helper functions ---
def rgb_to_lab_8u(img_rgb_uint8):
    lab = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)
    L = L.astype(np.float32) / 2.55  # scale back to [0,100]
    A = A.astype(np.float32) - 128.0
    B = B.astype(np.float32) - 128.0
    return L, A, B


def norm_LAB_for_net(L_uint8, A_uint8, B_uint8):
    # L = (L_uint8.astype(np.float32) / 255.0)
    # a = ((A_uint8.astype(np.float32) - 128.0) / 128.0)
    # b = ((B_uint8.astype(np.float32) - 128.0) / 128.0)
    # return L, a, b
    return L_uint8, A_uint8, B_uint8

# --- dataset ---
class ColorizationImageDataset(Dataset):
    def __init__(self, files, img_size=256):
        self.files = files
        self.size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        img = Image.open(path).convert("RGB")
        img = img.resize((self.size, self.size), Image.BICUBIC)
        rgb = np.array(img, dtype=np.uint8)
        L, A, B = rgb_to_lab_8u(rgb)
        Lf, af, bf = norm_LAB_for_net(L, A, B)
        L_t = torch.from_numpy(Lf).unsqueeze(0).float()
        ab_t = torch.from_numpy(np.stack([af, bf], 0)).float()
        return L_t, ab_t, path

# --- load files ---
from sklearn.model_selection import train_test_split
from pathlib import Path
# IMG_EXTS = {".jpg", ".jpeg", ".png"}
# DATA_ROOT = "/kaggle/input/coco-validation-2017/val2017"
# all_files = [str(p) for p in Path(DATA_ROOT).rglob("*") if p.suffix.lower() in IMG_EXTS]

# train_files, val_files = train_test_split(all_files, test_size=0.1, random_state=42)
# train_ds = ColorizationImageDataset(train_files, img_size=256)
# val_ds   = ColorizationImageDataset(val_files,   img_size=256)

# train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
# val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

TRAIN_ROOT = "/content/images/train/test2017"
VAL_ROOT   = "/content/images/val/val2017"

IMG_EXTS = {".jpg", ".jpeg", ".png"}

train_files = [str(p) for p in Path(TRAIN_ROOT).rglob("*") if p.suffix.lower() in IMG_EXTS]
val_files   = [str(p) for p in Path(VAL_ROOT).rglob("*") if p.suffix.lower() in IMG_EXTS]

train_ds = ColorizationImageDataset(train_files, img_size=256)
val_ds   = ColorizationImageDataset(val_files,   img_size=256)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)








import torch, torch.nn as nn, torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import os

# ------------------ ECCV 2016 Generator ------------------
class BaseColor(nn.Module):
    def __init__(self):
        super(BaseColor, self).__init__()
        self.l_cent = 50.
        self.l_norm = 100.
        self.ab_norm = 110.

    def normalize_l(self, in_l): return (in_l - self.l_cent) / self.l_norm
    def unnormalize_l(self, in_l): return in_l * self.l_norm + self.l_cent
    def normalize_ab(self, in_ab): return in_ab / self.ab_norm
    def unnormalize_ab(self, in_ab): return in_ab * self.ab_norm

class ECCVGenerator(BaseColor):
    def __init__(self, norm_layer=nn.BatchNorm2d):
        super(ECCVGenerator, self).__init__()

        self.model1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(64)
        )

        self.model2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(128)
        )

        self.model3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(256)
        )

        self.model4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )

        self.model5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )

        self.model6 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )

        self.model7 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )

        self.model8 = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 313, kernel_size=1, stride=1, padding=0, bias=True)
        )

        self.softmax = nn.Softmax(dim=1)
        self.model_out = nn.Conv2d(313, 2, kernel_size=1, padding=0, stride=1, bias=False)
        self.upsample4 = nn.Upsample(scale_factor=4, mode='bilinear')

    def forward(self, input_l):
        conv1_2 = self.model1(self.normalize_l(input_l))
        conv2_2 = self.model2(conv1_2)
        conv3_3 = self.model3(conv2_2)
        conv4_3 = self.model4(conv3_3)
        conv5_3 = self.model5(conv4_3)
        conv6_3 = self.model6(conv5_3)
        conv7_3 = self.model7(conv6_3)
        conv8_3 = self.model8(conv7_3)
        out_reg = self.model_out(self.softmax(conv8_3))
        return self.unnormalize_ab(self.upsample4(out_reg))

# ------------------ Training ------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 5
CKPT_DIR = "/content"
os.makedirs(CKPT_DIR, exist_ok=True)

model_e_pt = ECCVGenerator().to(DEVICE)

# ---- Load pretrained ECCV16 weights ----
weights = torch.hub.load_state_dict_from_url(
    'https://colorizers.s3.us-east-2.amazonaws.com/colorization_release_v2-9b330a0b.pth',
    map_location='cpu', check_hash=True
)
model_e_pt.load_state_dict(weights, strict=False)
print("Loaded pretrained ECCV16 weights.")

# ---- Optimizer ----
opt = torch.optim.AdamW(model_e_pt.parameters(), lr=1e-5, weight_decay=1e-4)
# scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
scaler = torch.amp.GradScaler("cuda", enabled=False)
best_val_loss = float("inf")

train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, EPOCHS + 1):
    model_e_pt.train()
    train_loss, train_acc = 0.0, 0.0
    for L, ab, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]"):
        L, ab = L.to(DEVICE), ab.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        # with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
        #     pred_ab = model(L)
        #     loss = F.mse_loss(pred_ab, ab)
        with torch.amp.autocast("cuda", enabled=False):
            pred_ab = model_e_pt(L)
            loss = F.mse_loss(pred_ab, ab)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        train_loss += loss.item() * L.size(0)
        train_acc += torch.mean((torch.abs(pred_ab - ab) < 10.0).float()).item()
    train_loss /= len(train_ds)
    train_acc /= len(train_loader)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # ---------- Validation ----------
    model_e_pt.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
        for L, ab, _ in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [val]"):
            L, ab = L.to(DEVICE), ab.to(DEVICE)
            # with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            #     pred_ab = model(L)
            #     loss = F.mse_loss(pred_ab, ab)
            with torch.amp.autocast("cuda", enabled=False):
                  pred_ab = model_e_pt(L)
                  loss = F.mse_loss(pred_ab, ab)
            val_loss += loss.item() * L.size(0)
            val_acc += torch.mean((torch.abs(pred_ab - ab) < 10.0).float()).item()
    val_loss /= len(val_ds)
    val_acc /= len(val_loader)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"\nEpoch {epoch}: Train L2 = {train_loss:.4f} | Train Acc = {train_acc*100:.2f}% | Val L2 = {val_loss:.4f} | Val Acc = {val_acc*100:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model_e_pt.state_dict(),
            "optimizer_state_dict": opt.state_dict(),
            "val_loss": val_loss,
        }, f"{CKPT_DIR}/best_eccv16_pretrained.pt")
        print(f"Saved new best model at epoch {epoch} (Val L2 = {val_loss:.4f} | Val Acc = {val_acc*100:.2f}%)")

print("\nTraining complete.")
print(f"Lowest validation loss: {best_val_loss:.4f}")

# ---------- Plot ----------
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(range(1,EPOCHS+1), train_losses, label="Train L2")
plt.plot(range(1,EPOCHS+1), val_losses, label="Val L2")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss vs Epoch"); plt.legend()

plt.subplot(1,2,2)
plt.plot(range(1,EPOCHS+1), train_accs, 'b-', label="Train Accuracy")
plt.plot(range(1,EPOCHS+1), val_accs, 'g-', label="Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Accuracy vs Epoch"); plt.legend()

plt.tight_layout()
plt.savefig(f"{CKPT_DIR}/eccv16_pretrained_curves.png")
print(f"Saved training graphs to {CKPT_DIR}/eccv16_pretrained_curves.png")







import torch, cv2, numpy as np
from PIL import Image
from skimage import color
import torch.nn.functional as F

# --- Load image (grayscale or color) ---
def load_image(img_path, size=256):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((size, size), Image.BICUBIC)
    rgb = np.array(img)
    lab = color.rgb2lab(rgb).astype("float32")
    L = lab[..., 0]
    ab = lab[..., 1:]
    return rgb, L, ab

# --- Convert LAB → RGB uint8 ---
def lab_to_rgb_uint8(L, ab):
  # L: (1,1,H,W) in 0..100 ; ab: (1,2,H,W) in original units
  L_ = L.squeeze().cpu().numpy()             # (H,W)
  ab_ = ab.squeeze().cpu().numpy()           # (2,H,W)
  lab = np.stack([L_, ab_[0], ab_[1]], axis=-1).astype(np.float32)  # (H,W,3)
  rgb = np.clip(color.lab2rgb(lab), 0, 1)
  return (rgb * 255).astype(np.uint8)

# --- Colorize with pretrained model ---
def colorize_image(model, img_path, size=256, device="cuda" if torch.cuda.is_available() else "cpu"):
    rgb, L, _ = load_image(img_path, size=size)
    L_t = torch.from_numpy(L).unsqueeze(0).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        pred_ab = model(L_t)                       # (1,2,H,W)
        pred_ab_up = F.interpolate(pred_ab, size=(L.shape[0], L.shape[1]), mode='bilinear')
        out_lab = torch.cat((L_t, pred_ab_up), dim=1)[0].cpu().numpy().transpose(1,2,0)
        out_rgb = np.clip(color.lab2rgb(out_lab), 0, 1)
        return (out_rgb * 255).astype(np.uint8)


# from skimage import color
# import numpy as np



# pred_ab = run_siggraph_forward(model, "/kaggle/input/coco-validation-2017/val2017/000000001268.jpg", 256)
# l_input, _ = prepare_siggraph_inputs("/kaggle/input/coco-validation-2017/val2017/000000001268.jpg", size=256, device=next(model.parameters()).device)
# rgb_uint8 = lab_to_rgb_uint8(l_input, pred_ab)

# Image.fromarray(rgb_uint8)




img_path = "/content/images/train/test2017/000000001286.jpg"
colored = colorize_image(model_e_pt, img_path, size=256)
Image.fromarray(colored).save("/content/output_colored_e_pt.png")
Image.fromarray(colored)

In [ ]:
# siggraph with pretrained weights

import torch
import torch.nn as nn

class BaseColor(nn.Module):
    def __init__(self):
        super(BaseColor, self).__init__()
        self.l_cent = 50.
        self.l_norm = 100.
        self.ab_norm = 110.

    def normalize_l(self, in_l):
        return (in_l - self.l_cent) / self.l_norm

    def unnormalize_ab(self, in_ab):
        return in_ab * self.ab_norm


class SIGGRAPHGenerator(BaseColor):
    def __init__(self, norm_layer=nn.BatchNorm2d, classes=529):
        super(SIGGRAPHGenerator, self).__init__()

        def C(in_c, out_c, k=3, s=1, p=1, d=1):
            return nn.Conv2d(in_c, out_c, k, s, p, dilation=d, bias=True)

        self.model1 = nn.Sequential(
            C(4,64), nn.ReLU(True),
            C(64,64), nn.ReLU(True),
            norm_layer(64)
        )
        self.model2 = nn.Sequential(
            C(64,128), nn.ReLU(True),
            C(128,128), nn.ReLU(True),
            norm_layer(128)
        )
        self.model3 = nn.Sequential(
            C(128,256), nn.ReLU(True),
            C(256,256), nn.ReLU(True),
            C(256,256), nn.ReLU(True),
            norm_layer(256)
        )
        self.model4 = nn.Sequential(
            C(256,512), nn.ReLU(True),
            C(512,512), nn.ReLU(True),
            C(512,512), nn.ReLU(True),
            norm_layer(512)
        )
        self.model5 = nn.Sequential(
            C(512,512,3,1,2,2), nn.ReLU(True),
            C(512,512,3,1,2,2), nn.ReLU(True),
            C(512,512,3,1,2,2), nn.ReLU(True),
            norm_layer(512)
        )
        self.model6 = nn.Sequential(
            C(512,512,3,1,2,2), nn.ReLU(True),
            C(512,512,3,1,2,2), nn.ReLU(True),
            C(512,512,3,1,2,2), nn.ReLU(True),
            norm_layer(512)
        )
        self.model7 = nn.Sequential(
            C(512,512), nn.ReLU(True),
            C(512,512), nn.ReLU(True),
            C(512,512), nn.ReLU(True),
            norm_layer(512)
        )

        # Decoder
        self.model8up = nn.Sequential(nn.ConvTranspose2d(512,256,4,2,1))
        self.model3short8 = nn.Sequential(C(256,256))
        self.model8 = nn.Sequential(
            nn.ReLU(True),
            C(256,256), nn.ReLU(True),
            C(256,256), nn.ReLU(True),
            norm_layer(256)
        )

        self.model9up = nn.Sequential(nn.ConvTranspose2d(256,128,4,2,1))
        self.model2short9 = nn.Sequential(C(128,128))
        self.model9 = nn.Sequential(
            nn.ReLU(True),
            C(128,128), nn.ReLU(True),
            norm_layer(128)
        )

        self.model10up = nn.Sequential(nn.ConvTranspose2d(128,128,4,2,1))
        self.model1short10 = nn.Sequential(C(64,128))
        self.model10 = nn.Sequential(
            nn.ReLU(True),
            C(128,128), nn.LeakyReLU(0.2, True)
        )

        self.model_out = nn.Sequential(
            C(128,2,1,1,0), nn.Tanh()
        )

    def forward(self, input_L, input_ab=None, mask=None):
        if input_ab is None:
            input_ab = torch.cat((input_L*0, input_L*0), dim=1)
        if mask is None:
            mask = input_L*0

        x = torch.cat((self.normalize_l(input_L), input_ab, mask), dim=1)
        c1 = self.model1(x)
        c2 = self.model2(c1[:,:,::2,::2])
        c3 = self.model3(c2[:,:,::2,::2])
        c4 = self.model4(c3[:,:,::2,::2])
        c5 = self.model5(c4)
        c6 = self.model6(c5)
        c7 = self.model7(c6)

        u8 = self.model8up(c7) + self.model3short8(c3)
        c8 = self.model8(u8)
        u9 = self.model9up(c8) + self.model2short9(c2)
        c9 = self.model9(u9)
        u10 = self.model10up(c9) + self.model1short10(c1)
        c10 = self.model10(u10)
        return self.unnormalize_ab(self.model_out(c10))



from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
import numpy as np
import torch
import cv2
from pathlib import Path

# --- your helper functions ---
def rgb_to_lab_8u(img_rgb_uint8):
    lab = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)
    L = L.astype(np.float32) / 2.55  # scale back to [0,100]
    A = A.astype(np.float32) - 128.0
    B = B.astype(np.float32) - 128.0
    return L, A, B


def norm_LAB_for_net(L_uint8, A_uint8, B_uint8):
    # L = (L_uint8.astype(np.float32) / 255.0)
    # a = ((A_uint8.astype(np.float32) - 128.0) / 128.0)
    # b = ((B_uint8.astype(np.float32) - 128.0) / 128.0)
    # return L, a, b
    return L_uint8, A_uint8, B_uint8

# --- dataset ---
class ColorizationImageDataset(Dataset):
    def __init__(self, files, img_size=256):
        self.files = files
        self.size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        img = Image.open(path).convert("RGB")
        img = img.resize((self.size, self.size), Image.BICUBIC)
        rgb = np.array(img, dtype=np.uint8)
        L, A, B = rgb_to_lab_8u(rgb)
        Lf, af, bf = norm_LAB_for_net(L, A, B)
        L_t = torch.from_numpy(Lf).unsqueeze(0).float()
        ab_t = torch.from_numpy(np.stack([af, bf], 0)).float()
        return L_t, ab_t, path

# --- load files ---
from sklearn.model_selection import train_test_split
from pathlib import Path
# IMG_EXTS = {".jpg", ".jpeg", ".png"}
# DATA_ROOT = "/kaggle/input/coco-validation-2017/val2017"
# all_files = [str(p) for p in Path(DATA_ROOT).rglob("*") if p.suffix.lower() in IMG_EXTS]

# train_files, val_files = train_test_split(all_files, test_size=0.1, random_state=42)
# train_ds = ColorizationImageDataset(train_files, img_size=256)
# val_ds   = ColorizationImageDataset(val_files,   img_size=256)

# train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
# val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

TRAIN_ROOT = "/content/images/train/test2017"
VAL_ROOT   = "/content/images/val/val2017"

IMG_EXTS = {".jpg", ".jpeg", ".png"}

train_files = [str(p) for p in Path(TRAIN_ROOT).rglob("*") if p.suffix.lower() in IMG_EXTS]
val_files   = [str(p) for p in Path(VAL_ROOT).rglob("*") if p.suffix.lower() in IMG_EXTS]

train_ds = ColorizationImageDataset(train_files, img_size=256)
val_ds   = ColorizationImageDataset(val_files,   img_size=256)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)



import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt

# ------------------ Training ------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 5
CKPT_DIR = "/content"
os.makedirs(CKPT_DIR, exist_ok=True)

model_s_pt = SIGGRAPHGenerator().to(DEVICE)

# ---- Load pretrained SIGGraph17 weights ----
weights = torch.hub.load_state_dict_from_url(
    'https://colorizers.s3.us-east-2.amazonaws.com/siggraph17-df00044c.pth',
    map_location='cpu', check_hash=True
)
model_s_pt.load_state_dict(weights, strict=False)
print("Loaded pretrained SIGGraph17 weights.")

# ---- Optimizer ----
opt = torch.optim.AdamW(model_s_pt.parameters(), lr=1e-5, weight_decay=1e-4)
# scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
scaler = torch.amp.GradScaler("cuda", enabled=False)
best_val_loss = float("inf")

train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, EPOCHS + 1):
    model_s_pt.train()
    train_loss, train_acc = 0.0, 0.0
    for L, ab, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]"):
        L, ab = L.to(DEVICE), ab.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            pred_ab = model_s_pt(L)
            loss = F.mse_loss(pred_ab, ab)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        train_loss += loss.item() * L.size(0)
        train_acc += torch.mean((torch.abs(pred_ab - ab) < 10.0).float()).item()
    train_loss /= len(train_ds)
    train_acc /= len(train_loader)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # ---------- Validation ----------
    model_s_pt.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
        for L, ab, _ in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [val]"):
            L, ab = L.to(DEVICE), ab.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                pred_ab = model_s_pt(L)
                loss = F.mse_loss(pred_ab, ab)
            val_loss += loss.item() * L.size(0)
            val_acc += torch.mean((torch.abs(pred_ab - ab) < 10.0).float()).item()
    val_loss /= len(val_ds)
    val_acc /= len(val_loader)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"\nEpoch {epoch}: Train L2 = {train_loss:.4f} | Train Acc = {train_acc*100:.2f}% | Val L2 = {val_loss:.4f} | Val Acc = {val_acc*100:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model_s_pt.state_dict(),
            "optimizer_state_dict": opt.state_dict(),
            "val_loss": val_loss,
        }, f"{CKPT_DIR}/best_sig17_pretrained.pt")
        print(f"Saved new best model at epoch {epoch} (Val L2 = {val_loss:.4f} | Val Acc = {val_acc*100:.2f}%)")

print("\nTraining complete.")
print(f"Lowest validation loss: {best_val_loss:.4f}")

# ---------- Plot ----------
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(range(1,EPOCHS+1), train_losses, label="Train L2")
plt.plot(range(1,EPOCHS+1), val_losses, label="Val L2")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss vs Epoch"); plt.legend()

plt.subplot(1,2,2)
plt.plot(range(1,EPOCHS+1), train_accs, 'b-', label="Train Accuracy")
plt.plot(range(1,EPOCHS+1), val_accs, 'g-', label="Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Accuracy vs Epoch"); plt.legend()

plt.tight_layout()
plt.savefig(f"{CKPT_DIR}/sig17_pretrained_curves.png")
print(f"Saved training graphs to {CKPT_DIR}/sig17_pretrained_curves.png")



import torch, cv2, numpy as np
from PIL import Image
from skimage import color
import torch.nn.functional as F

# --- Load image (grayscale or color) ---
def load_image(img_path, size=256):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((size, size), Image.BICUBIC)
    rgb = np.array(img)
    lab = color.rgb2lab(rgb).astype("float32")
    L = lab[..., 0]
    ab = lab[..., 1:]
    return rgb, L, ab

# --- Convert LAB → RGB uint8 ---
def lab_to_rgb_uint8(L, ab):
  # L: (1,1,H,W) in 0..100 ; ab: (1,2,H,W) in original units
  L_ = L.squeeze().cpu().numpy()             # (H,W)
  ab_ = ab.squeeze().cpu().numpy()           # (2,H,W)
  lab = np.stack([L_, ab_[0], ab_[1]], axis=-1).astype(np.float32)  # (H,W,3)
  rgb = np.clip(color.lab2rgb(lab), 0, 1)
  return (rgb * 255).astype(np.uint8)

# --- Colorize with pretrained model ---
def colorize_image(model, img_path, size=256, device="cuda" if torch.cuda.is_available() else "cpu"):
    rgb, L, _ = load_image(img_path, size=size)
    L_t = torch.from_numpy(L).unsqueeze(0).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        pred_ab = model(L_t)                       # (1,2,H,W)
        pred_ab_up = F.interpolate(pred_ab, size=(L.shape[0], L.shape[1]), mode='bilinear')
        out_lab = torch.cat((L_t, pred_ab_up), dim=1)[0].cpu().numpy().transpose(1,2,0)
        out_rgb = np.clip(color.lab2rgb(out_lab), 0, 1)
        return (out_rgb * 255).astype(np.uint8)


# from skimage import color
# import numpy as np



# pred_ab = run_siggraph_forward(model, "/kaggle/input/coco-validation-2017/val2017/000000001268.jpg", 256)
# l_input, _ = prepare_siggraph_inputs("/kaggle/input/coco-validation-2017/val2017/000000001268.jpg", size=256, device=next(model.parameters()).device)
# rgb_uint8 = lab_to_rgb_uint8(l_input, pred_ab)

# Image.fromarray(rgb_uint8)




img_path = "/content/images/train/test2017/000000001286.jpg"
colored = colorize_image(model_s_pt, img_path, size=256)
Image.fromarray(colored).save("/content/output_colored_s_pt.png")
Image.fromarray(colored)

In [ ]:
# ensemble for photos

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from skimage import color
from tqdm import tqdm # Make sure tqdm is imported if used elsewhere
import os

# ---------------- 1. DEFINE BASE CLASS ----------------
# (Using modern Python 3 'super()')
class BaseColor(nn.Module):
    def __init__(self):
        super().__init__() # Use modern super()
        self.l_cent = 50.
        self.l_norm = 100.
        self.ab_norm = 110.

    def normalize_l(self, in_l): return (in_l - self.l_cent) / self.l_norm
    def unnormalize_l(self, in_l): return in_l * self.l_norm + self.l_cent
    def normalize_ab(self, in_ab): return in_ab / self.ab_norm
    def unnormalize_ab(self, in_ab): return in_ab * self.ab_norm

# ---------------- 2. DEFINE ECCVGenerator ----------------
class ECCVGenerator(BaseColor):
    def __init__(self, norm_layer=nn.BatchNorm2d):
        super().__init__()

        self.model1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(64)
        )
        self.model2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(128)
        )
        self.model3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(256)
        )
        self.model4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model6 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model7 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model8 = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 313, kernel_size=1, stride=1, padding=0, bias=True)
        )
        self.softmax = nn.Softmax(dim=1)
        self.model_out = nn.Conv2d(313, 2, kernel_size=1, padding=0, stride=1, bias=False)
        self.upsample4 = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False)

    def forward(self, input_l):
        conv1_2 = self.model1(self.normalize_l(input_l))
        conv2_2 = self.model2(conv1_2)
        conv3_3 = self.model3(conv2_2)
        conv4_3 = self.model4(conv3_3)
        conv5_3 = self.model5(conv4_3)
        conv6_3 = self.model6(conv5_3)
        conv7_3 = self.model7(conv6_3)
        conv8_3 = self.model8(conv7_3)
        out_reg = self.model_out(self.softmax(conv8_3))
        return self.unnormalize_ab(self.upsample4(out_reg))

# ---------------- 3. DEFINE SIGGRAPHGenerator ----------------
class SIGGRAPHGenerator(BaseColor):
    def __init__(self, norm_layer=nn.BatchNorm2d, classes=529):
        super().__init__()

        # Re-using the clean definition from your previous script
        def C(in_c, out_c, k=3, s=1, p=1, d=1):
            return nn.Conv2d(in_c, out_c, k, s, p, dilation=d, bias=True)

        self.model1 = nn.Sequential(C(4,64), nn.ReLU(True), C(64,64), nn.ReLU(True), norm_layer(64))
        self.model2 = nn.Sequential(C(64,128), nn.ReLU(True), C(128,128), nn.ReLU(True), norm_layer(128))
        self.model3 = nn.Sequential(C(128,256), nn.ReLU(True), C(256,256), nn.ReLU(True), C(256,256), nn.ReLU(True), norm_layer(256))
        self.model4 = nn.Sequential(C(256,512), nn.ReLU(True), C(512,512), nn.ReLU(True), C(512,512), nn.ReLU(True), norm_layer(512))
        self.model5 = nn.Sequential(C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), norm_layer(512))
        self.model6 = nn.Sequential(C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), norm_layer(512))
        self.model7 = nn.Sequential(C(512,512), nn.ReLU(True), C(512,512), nn.ReLU(True), C(512,512), nn.ReLU(True), norm_layer(512))

        self.model8up = nn.Sequential(nn.ConvTranspose2d(512,256,4,2,1))
        self.model3short8 = nn.Sequential(C(256,256))
        self.model8 = nn.Sequential(nn.ReLU(True), C(256,256), nn.ReLU(True), C(256,256), nn.ReLU(True), norm_layer(256))

        self.model9up = nn.Sequential(nn.ConvTranspose2d(256,128,4,2,1))
        self.model2short9 = nn.Sequential(C(128,128))
        self.model9 = nn.Sequential(nn.ReLU(True), C(128,128), nn.ReLU(True), norm_layer(128))

        self.model10up = nn.Sequential(nn.ConvTranspose2d(128,128,4,2,1))
        self.model1short10 = nn.Sequential(C(64,128))
        self.model10 = nn.Sequential(nn.ReLU(True), C(128,128), nn.LeakyReLU(0.2, True))

        self.model_out = nn.Sequential(C(128,2,1,1,0), nn.Tanh())

    def forward(self, input_L, input_ab=None, mask=None):
        if input_ab is None:
            input_ab = torch.cat((input_L*0, input_L*0), dim=1)
        if mask is None:
            mask = input_L*0

        x = torch.cat((self.normalize_l(input_L), self.normalize_ab(input_ab), mask), dim=1)
        c1 = self.model1(x)
        c2 = self.model2(c1[:,:,::2,::2])
        c3 = self.model3(c2[:,:,::2,::2])
        c4 = self.model4(c3[:,:,::2,::2])
        c5 = self.model5(c4)
        c6 = self.model6(c5)
        c7 = self.model7(c6)

        u8 = self.model8up(c7) + self.model3short8(c3)
        c8 = self.model8(u8)
        u9 = self.model9up(c8) + self.model2short9(c2)
        c9 = self.model9(u9)
        u10 = self.model10up(c9) + self.model1short10(c1)
        c10 = self.model10(u10)
        return self.unnormalize_ab(self.model_out(c10))

# ---------------- 4. HELPER FUNCTIONS ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_image(img_path, size=256):
    """Loads and processes an image for the model."""
    img = Image.open(img_path).convert("RGB")
    img = img.resize((size, size), Image.BICUBIC)
    rgb = np.array(img)
    lab = color.rgb2lab(rgb).astype("float32")
    L = lab[..., 0]
    ab = lab[..., 1:]
    return rgb, L, ab

# ---------------- 5. LOAD MODELS ----------------
# --- Load ECCV16 ---
model_e_pt = ECCVGenerator().to(DEVICE)
ckpt_e = torch.load("/content/best_eccv16_pretrained.pt", map_location=DEVICE)
model_e_pt.load_state_dict(ckpt_e["model_state_dict"] if "model_state_dict" in ckpt_e else ckpt_e)
print("Loaded pretrained ECCV16 model")

# --- Load SIGGRAPH17 ---
model_s_pt = SIGGRAPHGenerator().to(DEVICE)
ckpt_s = torch.load("/content/best_sig17_pretrained.pt", map_location=DEVICE)
model_s_pt.load_state_dict(ckpt_s["model_state_dict"] if "model_state_dict" in ckpt_s else ckpt_s)
print("Loaded pretrained SIGGRAPH17 model")

# ---------------- 6. ENSEMBLE FUNCTION (FIXED) ----------------
def colorize_image_ensemble(model1, model2, img_path, size=256, device=DEVICE):
    rgb, L, _ = load_image(img_path, size=size)
    L_t = torch.from_numpy(L).unsqueeze(0).unsqueeze(0).to(device) # Shape: [1, 1, 256, 256]

    model1.eval()
    model2.eval()

    with torch.no_grad():
        pred_ab_1 = model1(L_t) # Outputs [1, 2, 256, 256]
        pred_ab_2 = model2(L_t) # Outputs [1, 2, 256, 256]

        # --- ensemble of ab outputs ---
        # Your weighted average:
        pred_ab = 0.4 * pred_ab_1 + 0.6 * pred_ab_2

        # --- BUG FIX: Remove unnecessary interpolate ---
        # Both models already output 256x256, so pred_ab is [1, 2, 256, 256]
        # pred_ab_up = torch.nn.functional.interpolate(pred_ab, size=(L.shape[0], L.shape[1]), mode='bilinear') # <-- REMOVED

        # Concatenate original L tensor with the final predicted ab tensor
        out_lab = torch.cat((L_t, pred_ab), dim=1)[0].cpu().numpy().transpose(1,2,0)
        out_rgb = np.clip(color.lab2rgb(out_lab), 0, 1)
        return (out_rgb * 255).astype(np.uint8), (rgb)

# ---------------- 7. RUN INFERENCE ----------------
img_path = "/content/000000537153-gray.jpg" # Make sure this file exists
colored, orig = colorize_image_ensemble(model_e_pt, model_s_pt, img_path, size=256)

# Save output
out_path = "/content/output_colored_ensemble_3.png"
Image.fromarray(colored).save(out_path)
print(f"Saved ensemble colorized image at {out_path}")

# --- Display side by side ---
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
# We use the 'L' channel (grayscale) from 'load_image' for the "Original"
# but since 'orig' is the RGB loaded, we'll convert it to gray for a true comparison
plt.imshow(color.rgb2gray(orig), cmap='gray')
plt.title("Original (Grayscale Input)")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(colored)
plt.title("Ensemble Colorized Output (ECCV16 + SIGGRAPH17)")
plt.axis("off")

plt.show()

In [ ]:
# ensemble vs stock model accuracies
# ensemble for photos

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from skimage import color
from tqdm import tqdm # Make sure tqdm is imported if used elsewhere
import os

# ---------------- 1. DEFINE BASE CLASS ----------------
# (Using modern Python 3 'super()')
class BaseColor(nn.Module):
    def __init__(self):
        super().__init__() # Use modern super()
        self.l_cent = 50.
        self.l_norm = 100.
        self.ab_norm = 110.

    def normalize_l(self, in_l): return (in_l - self.l_cent) / self.l_norm
    def unnormalize_l(self, in_l): return in_l * self.l_norm + self.l_cent
    def normalize_ab(self, in_ab): return in_ab / self.ab_norm
    def unnormalize_ab(self, in_ab): return in_ab * self.ab_norm

# ---------------- 2. DEFINE ECCVGenerator ----------------
class ECCVGenerator(BaseColor):
    def __init__(self, norm_layer=nn.BatchNorm2d):
        super().__init__()

        self.model1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(64)
        )
        self.model2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(128)
        )
        self.model3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(256)
        )
        self.model4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model6 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model7 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model8 = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 313, kernel_size=1, stride=1, padding=0, bias=True)
        )
        self.softmax = nn.Softmax(dim=1)
        self.model_out = nn.Conv2d(313, 2, kernel_size=1, padding=0, stride=1, bias=False)
        self.upsample4 = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False)

    def forward(self, input_l):
        conv1_2 = self.model1(self.normalize_l(input_l))
        conv2_2 = self.model2(conv1_2)
        conv3_3 = self.model3(conv2_2)
        conv4_3 = self.model4(conv3_3)
        conv5_3 = self.model5(conv4_3)
        conv6_3 = self.model6(conv5_3)
        conv7_3 = self.model7(conv6_3)
        conv8_3 = self.model8(conv7_3)
        out_reg = self.model_out(self.softmax(conv8_3))
        return self.unnormalize_ab(self.upsample4(out_reg))

# ---------------- 3. DEFINE SIGGRAPHGenerator ----------------
class SIGGRAPHGenerator(BaseColor):
    def __init__(self, norm_layer=nn.BatchNorm2d, classes=529):
        super().__init__()

        # Re-using the clean definition from your previous script
        def C(in_c, out_c, k=3, s=1, p=1, d=1):
            return nn.Conv2d(in_c, out_c, k, s, p, dilation=d, bias=True)

        self.model1 = nn.Sequential(C(4,64), nn.ReLU(True), C(64,64), nn.ReLU(True), norm_layer(64))
        self.model2 = nn.Sequential(C(64,128), nn.ReLU(True), C(128,128), nn.ReLU(True), norm_layer(128))
        self.model3 = nn.Sequential(C(128,256), nn.ReLU(True), C(256,256), nn.ReLU(True), C(256,256), nn.ReLU(True), norm_layer(256))
        self.model4 = nn.Sequential(C(256,512), nn.ReLU(True), C(512,512), nn.ReLU(True), C(512,512), nn.ReLU(True), norm_layer(512))
        self.model5 = nn.Sequential(C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), norm_layer(512))
        self.model6 = nn.Sequential(C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), norm_layer(512))
        self.model7 = nn.Sequential(C(512,512), nn.ReLU(True), C(512,512), nn.ReLU(True), C(512,512), nn.ReLU(True), norm_layer(512))

        self.model8up = nn.Sequential(nn.ConvTranspose2d(512,256,4,2,1))
        self.model3short8 = nn.Sequential(C(256,256))
        self.model8 = nn.Sequential(nn.ReLU(True), C(256,256), nn.ReLU(True), C(256,256), nn.ReLU(True), norm_layer(256))

        self.model9up = nn.Sequential(nn.ConvTranspose2d(256,128,4,2,1))
        self.model2short9 = nn.Sequential(C(128,128))
        self.model9 = nn.Sequential(nn.ReLU(True), C(128,128), nn.ReLU(True), norm_layer(128))

        self.model10up = nn.Sequential(nn.ConvTranspose2d(128,128,4,2,1))
        self.model1short10 = nn.Sequential(C(64,128))
        self.model10 = nn.Sequential(nn.ReLU(True), C(128,128), nn.LeakyReLU(0.2, True))

        self.model_out = nn.Sequential(C(128,2,1,1,0), nn.Tanh())

    def forward(self, input_L, input_ab=None, mask=None):
        if input_ab is None:
            input_ab = torch.cat((input_L*0, input_L*0), dim=1)
        if mask is None:
            mask = input_L*0

        x = torch.cat((self.normalize_l(input_L), self.normalize_ab(input_ab), mask), dim=1)
        c1 = self.model1(x)
        c2 = self.model2(c1[:,:,::2,::2])
        c3 = self.model3(c2[:,:,::2,::2])
        c4 = self.model4(c3[:,:,::2,::2])
        c5 = self.model5(c4)
        c6 = self.model6(c5)
        c7 = self.model7(c6)

        u8 = self.model8up(c7) + self.model3short8(c3)
        c8 = self.model8(u8)
        u9 = self.model9up(c8) + self.model2short9(c2)
        c9 = self.model9(u9)
        u10 = self.model10up(c9) + self.model1short10(c1)
        c10 = self.model10(u10)
        return self.unnormalize_ab(self.model_out(c10))

# ---------------- 4. HELPER FUNCTIONS ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_image(img_path, size=256):
    """Loads and processes an image for the model."""
    img = Image.open(img_path).convert("RGB")
    img = img.resize((size, size), Image.BICUBIC)
    rgb = np.array(img)
    lab = color.rgb2lab(rgb).astype("float32")
    L = lab[..., 0]
    ab = lab[..., 1:]
    return rgb, L, ab

# ---------------- 5. LOAD MODELS ----------------
# --- Load ECCV16 ---
model_e_pt = ECCVGenerator().to(DEVICE)
ckpt_e = torch.load("/content/best_eccv16_pretrained.pt", map_location=DEVICE)
model_e_pt.load_state_dict(ckpt_e["model_state_dict"] if "model_state_dict" in ckpt_e else ckpt_e)
print("Loaded pretrained ECCV16 model")

# --- Load SIGGRAPH17 ---
model_s_pt = SIGGRAPHGenerator().to(DEVICE)
ckpt_s = torch.load("/content/best_sig17_pretrained.pt", map_location=DEVICE)
model_s_pt.load_state_dict(ckpt_s["model_state_dict"] if "model_state_dict" in ckpt_s else ckpt_s)
print("Loaded pretrained SIGGRAPH17 model")

# ---------------- 6. ENSEMBLE FUNCTION (FIXED) ----------------
def colorize_image_ensemble(model1, model2, img_path, size=256, device=DEVICE):
    rgb, L, _ = load_image(img_path, size=size)
    L_t = torch.from_numpy(L).unsqueeze(0).unsqueeze(0).to(device) # Shape: [1, 1, 256, 256]

    model1.eval()
    model2.eval()

    with torch.no_grad():
        pred_ab_1 = model1(L_t) # Outputs [1, 2, 256, 256]
        pred_ab_2 = model2(L_t) # Outputs [1, 2, 256, 256]

        # --- ensemble of ab outputs ---
        # Your weighted average:
        pred_ab = 0.4 * pred_ab_1 + 0.6 * pred_ab_2

        # --- BUG FIX: Remove unnecessary interpolate ---
        # Both models already output 256x256, so pred_ab is [1, 2, 256, 256]
        # pred_ab_up = torch.nn.functional.interpolate(pred_ab, size=(L.shape[0], L.shape[1]), mode='bilinear') # <-- REMOVED

        # Concatenate original L tensor with the final predicted ab tensor
        out_lab = torch.cat((L_t, pred_ab), dim=1)[0].cpu().numpy().transpose(1,2,0)
        out_rgb = np.clip(color.lab2rgb(out_lab), 0, 1)
        return (out_rgb * 255).astype(np.uint8), (rgb)


# ---------------- 8. VALIDATION DATASET SETUP ----------------

# --- Helper function from your training script ---
def rgb_to_lab_8u(img_rgb_uint8):
    lab = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)
    L = L.astype(np.float32) / 2.55  # scale back to [0,100]
    A = A.astype(np.float32) - 128.0
    B = B.astype(np.float32) - 128.0
    return L, A, B

# --- Dataset class from your training script ---
class ColorizationImageDataset(Dataset):
    def __init__(self, files, img_size=256):
        self.files = files
        self.size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        try:
            img = Image.open(path).convert("RGB")
            img = img.resize((self.size, self.size), Image.BICUBIC)
            rgb = np.array(img, dtype=np.uint8)
            L, A, B = rgb_to_lab_8u(rgb)
            L_t = torch.from_numpy(L).unsqueeze(0).float()
            ab_t = torch.from_numpy(np.stack([A, B], 0)).float()
            return L_t, ab_t
        except Exception as e:
            print(f"Warning: Skipping file {path} due to error: {e}")
            return None # Will be filtered by collate_fn

# --- Custom collate function to filter Nones ---
def collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    return torch.utils.data.dataloader.default_collate(batch) if batch else (None, None)

# --- Setup DataLoader (assuming same path as training) ---
VAL_ROOT = "/content/images/val/val2017"
IMG_EXTS = {".jpg", ".jpeg", ".png"}

try:
    val_files = [str(p) for p in Path(VAL_ROOT).rglob("*") if p.suffix.lower() in IMG_EXTS]
    if not val_files:
        print(f"Warning: No validation files found at {VAL_ROOT}. Skipping validation.")
        val_loader = None
    else:
        val_ds = ColorizationImageDataset(val_files, img_size=256)
        # Use batch_size 16-32 for validation
        val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
        print(f"Loaded {len(val_files)} validation files.")
except FileNotFoundError:
    print(f"Error: Validation directory not found at {VAL_ROOT}. Skipping validation.")
    val_loader = None


# ---------------- 9. FULL VALIDATION SCRIPT ----------------

if val_loader:
    print("\nStarting validation on the full dataset...")

    # Set models to evaluation mode
    model_e_pt.eval()
    model_s_pt.eval()

    # Trackers for all 3 models
    total_loss_e, total_loss_s, total_loss_ens = 0.0, 0.0, 0.0
    total_acc_e, total_acc_s, total_acc_ens = 0.0, 0.0, 0.0

    with torch.no_grad():
        for L, ab in tqdm(val_loader, desc="Validating"):
            # Filter out empty batches from collate_fn
            if L is None or ab is None:
                continue

            L, ab = L.to(DEVICE), ab.to(DEVICE)

            # 1. Get predictions from both models
            pred_ab_e = model_e_pt(L)
            pred_ab_s = model_s_pt(L)

            # 2. Get ensemble prediction
            pred_ab_ens = 0.4 * pred_ab_e + 0.6 * pred_ab_s

            # 3. Calculate Loss (using L2/MSE as in your training)
            # .item() gets the scalar value from the tensor
            total_loss_e += F.mse_loss(pred_ab_e, ab).item()
            total_loss_s += F.mse_loss(pred_ab_s, ab).item()
            total_loss_ens += F.mse_loss(pred_ab_ens, ab).item()

            # 4. Calculate Accuracy (using your <10 unit metric)
            total_acc_e += torch.mean((torch.abs(pred_ab_e - ab) < 10.0).float()).item()
            total_acc_s += torch.mean((torch.abs(pred_ab_s - ab) < 10.0).float()).item()
            total_acc_ens += torch.mean((torch.abs(pred_ab_ens - ab) < 10.0).float()).item()

    # 5. Average results
    num_batches = len(val_loader)
    avg_loss_e = total_loss_e / num_batches
    avg_loss_s = total_loss_s / num_batches
    avg_loss_ens = total_loss_ens / num_batches

    avg_acc_e = (total_acc_e / num_batches) * 100
    avg_acc_s = (total_acc_s / num_batches) * 100
    avg_acc_ens = (total_acc_ens / num_batches) * 100

    # 6. Print the final report
    print("\n" + "="*30)
    print("  Full Validation Report")
    print("="*30)
    print(f"  Validation Set Size: {len(val_ds)} images")
    print(f"  Batch Size: 16, Batches: {num_batches}")
    print("\n--- Average L2 Loss (MSE) ---")
    print(f"  ECCV16:     {avg_loss_e:.4f}")
    print(f"  SIGGRAPH17: {avg_loss_s:.4f}")
    print(f"  Ensemble:   {avg_loss_ens:.4f}")
    print("\n--- Average Accuracy (<10 LAB units) ---")
    print(f"  ECCV16:     {avg_acc_e:.2f}%")
    print(f"  SIGGRAPH17: {avg_acc_s:.2f}%")
    print(f"  Ensemble:   {avg_acc_ens:.2f}%")
    print("="*30)
else:
    print("Validation loader not initialized. Cannot run full validation.")



In [ ]:
# ensemble for videos

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from skimage import color
from tqdm import tqdm
import os
import cv2 # <-- ADD THIS IMPORT

# ---------------- 8. NEW FUNCTIONS FOR VIDEO ----------------

def process_frame(rgb_frame, size=256):
    """Loads and processes an in-memory BGR frame for the model."""
    # Convert BGR (OpenCV default) to RGB
    rgb = cv2.cvtColor(rgb_frame, cv2.COLOR_BGR2RGB)

    # Resize using OpenCV, matching PIL's BICUBIC
    rgb = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_CUBIC)

    # Process like load_image
    lab = color.rgb2lab(rgb).astype("float32")
    L = lab[..., 0]
    return rgb, L

def colorize_frame_ensemble(model1, model2, bgr_frame, size=256, device=DEVICE):
    """Runs the ensemble on a single BGR video frame."""

    # 1. Process the frame (resize, convert to LAB, get L channel)
    rgb, L = process_frame(bgr_frame, size=size)
    L_t = torch.from_numpy(L).unsqueeze(0).unsqueeze(0).to(device) # Shape: [1, 1, 256, 256]

    # Models are already in eval mode from file loading, but good practice:
    model1.eval()
    model2.eval()

    with torch.no_grad():
        pred_ab_1 = model1(L_t) # Outputs [1, 2, 256, 256]
        pred_ab_2 = model2(L_t) # Outputs [1, 2, 256, 256]

        # 2. Ensemble the 'ab' outputs
        pred_ab = 0.4 * pred_ab_1 + 0.6 * pred_ab_2

        # 3. Combine with original L tensor
        out_lab = torch.cat((L_t, pred_ab), dim=1)[0].cpu().numpy().transpose(1,2,0)

        # 4. Convert back to RGB
        out_rgb_float = np.clip(color.lab2rgb(out_lab), 0, 1)
        out_rgb_uint8 = (out_rgb_float * 255).astype(np.uint8)

        # 5. Convert back to BGR for OpenCV
        out_bgr = cv2.cvtColor(out_rgb_uint8, cv2.COLOR_RGB2BGR)

        return out_bgr

# ---------------- 9. RUN VIDEO COLORIZATION (FIXED) ----------------

# --- Define your video paths ---
VIDEO_IN_PATH = "/content/Untitled video - Made with Clipchamp (2).mp4"
VIDEO_OUT_PATH = "/content/output_video_colorized_3.mp4"

# --- Open the input video ---
cap = cv2.VideoCapture(VIDEO_IN_PATH)
if not cap.isOpened():
    print(f"Error: Could not open video file {VIDEO_IN_PATH}")
else:
    # Get video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # --- CHANGE 1: Define output size using original dimensions ---
    output_size = (frame_width, frame_height)

    # --- CHANGE 2: Use original output_size for the writer ---
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for .mp4
    out = cv2.VideoWriter(VIDEO_OUT_PATH, fourcc, fps, output_size)

    # We still process at 256x256, but the output video will be the original size
    processing_size = 256
    print(f"Processing video: {VIDEO_IN_PATH}")
    print(f"Original Res: {frame_width}x{frame_height}, Processing Res: {processing_size}x{processing_size}")

    # Use tqdm for a progress bar
    for _ in tqdm(range(frame_count), desc="Colorizing frames"):
        ret, frame = cap.read()
        if not ret:
            break # End of video

        # Colorize the frame at 256x256
        # colorized_frame is (256, 256)
        colorized_frame = colorize_frame_ensemble(model_e_pt, model_s_pt, frame, size=processing_size)

        # --- CHANGE 3: Resize the (256, 256) output back to the original video size ---
        final_frame = cv2.resize(colorized_frame, output_size, interpolation=cv2.INTER_CUBIC)

        # --- CHANGE 4: Write the resized final_frame ---
        out.write(final_frame)

    # Release everything when job is finished
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"\nVideo processing complete. Saved to: {VIDEO_OUT_PATH}")

In [ ]:
# ensemble for photos (aspect ratio corrected)
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from skimage import color
from tqdm import tqdm # Make sure tqdm is imported if used elsewhere
import os

# ---------------- 1. DEFINE BASE CLASS ----------------
# (Using modern Python 3 'super()')
class BaseColor(nn.Module):
    def __init__(self):
        super().__init__() # Use modern super()
        self.l_cent = 50.
        self.l_norm = 100.
        self.ab_norm = 110.

    def normalize_l(self, in_l): return (in_l - self.l_cent) / self.l_norm
    def unnormalize_l(self, in_l): return in_l * self.l_norm + self.l_cent
    def normalize_ab(self, in_ab): return in_ab / self.ab_norm
    def unnormalize_ab(self, in_ab): return in_ab * self.ab_norm

# ---------------- 2. DEFINE ECCVGenerator ----------------
class ECCVGenerator(BaseColor):
    def __init__(self, norm_layer=nn.BatchNorm2d):
        super().__init__()

        self.model1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(64)
        )
        self.model2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(128)
        )
        self.model3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(256)
        )
        self.model4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model6 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, dilation=2, stride=1, padding=2, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model7 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            norm_layer(512)
        )
        self.model8 = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.ReLU(True),
            nn.Conv2d(256, 313, kernel_size=1, stride=1, padding=0, bias=True)
        )
        self.softmax = nn.Softmax(dim=1)
        self.model_out = nn.Conv2d(313, 2, kernel_size=1, padding=0, stride=1, bias=False)
        self.upsample4 = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False)

    def forward(self, input_l):
        conv1_2 = self.model1(self.normalize_l(input_l))
        conv2_2 = self.model2(conv1_2)
        conv3_3 = self.model3(conv2_2)
        conv4_3 = self.model4(conv3_3)
        conv5_3 = self.model5(conv4_3)
        conv6_3 = self.model6(conv5_3)
        conv7_3 = self.model7(conv6_3)
        conv8_3 = self.model8(conv7_3)
        out_reg = self.model_out(self.softmax(conv8_3))
        return self.unnormalize_ab(self.upsample4(out_reg))

# ---------------- 3. DEFINE SIGGRAPHGenerator ----------------
class SIGGRAPHGenerator(BaseColor):
    def __init__(self, norm_layer=nn.BatchNorm2d, classes=529):
        super().__init__()

        # Re-using the clean definition from your previous script
        def C(in_c, out_c, k=3, s=1, p=1, d=1):
            return nn.Conv2d(in_c, out_c, k, s, p, dilation=d, bias=True)

        self.model1 = nn.Sequential(C(4,64), nn.ReLU(True), C(64,64), nn.ReLU(True), norm_layer(64))
        self.model2 = nn.Sequential(C(64,128), nn.ReLU(True), C(128,128), nn.ReLU(True), norm_layer(128))
        self.model3 = nn.Sequential(C(128,256), nn.ReLU(True), C(256,256), nn.ReLU(True), C(256,256), nn.ReLU(True), norm_layer(256))
        self.model4 = nn.Sequential(C(256,512), nn.ReLU(True), C(512,512), nn.ReLU(True), C(512,512), nn.ReLU(True), norm_layer(512))
        self.model5 = nn.Sequential(C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), norm_layer(512))
        self.model6 = nn.Sequential(C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), C(512,512,3,1,2,2), nn.ReLU(True), norm_layer(512))
        self.model7 = nn.Sequential(C(512,512), nn.ReLU(True), C(512,512), nn.ReLU(True), C(512,512), nn.ReLU(True), norm_layer(512))

        self.model8up = nn.Sequential(nn.ConvTranspose2d(512,256,4,2,1))
        self.model3short8 = nn.Sequential(C(256,256))
        self.model8 = nn.Sequential(nn.ReLU(True), C(256,256), nn.ReLU(True), C(256,256), nn.ReLU(True), norm_layer(256))

        self.model9up = nn.Sequential(nn.ConvTranspose2d(256,128,4,2,1))
        self.model2short9 = nn.Sequential(C(128,128))
        self.model9 = nn.Sequential(nn.ReLU(True), C(128,128), nn.ReLU(True), norm_layer(128))

        self.model10up = nn.Sequential(nn.ConvTranspose2d(128,128,4,2,1))
        self.model1short10 = nn.Sequential(C(64,128))
        self.model10 = nn.Sequential(nn.ReLU(True), C(128,128), nn.LeakyReLU(0.2, True))

        self.model_out = nn.Sequential(C(128,2,1,1,0), nn.Tanh())

    def forward(self, input_L, input_ab=None, mask=None):
        if input_ab is None:
            input_ab = torch.cat((input_L*0, input_L*0), dim=1)
        if mask is None:
            mask = input_L*0

        x = torch.cat((self.normalize_l(input_L), self.normalize_ab(input_ab), mask), dim=1)
        c1 = self.model1(x)
        c2 = self.model2(c1[:,:,::2,::2])
        c3 = self.model3(c2[:,:,::2,::2])
        c4 = self.model4(c3[:,:,::2,::2])
        c5 = self.model5(c4)
        c6 = self.model6(c5)
        c7 = self.model7(c6)

        u8 = self.model8up(c7) + self.model3short8(c3)
        c8 = self.model8(u8)
        u9 = self.model9up(c8) + self.model2short9(c2)
        c9 = self.model9(u9)
        u10 = self.model10up(c9) + self.model1short10(c1)
        c10 = self.model10(u10)
        return self.unnormalize_ab(self.model_out(c10))

# ---------------- 4. HELPER FUNCTIONS (REVISED) ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_image(img_path, size=256):
    """Loads and processes an image for the model, preserving aspect ratio."""
    img = Image.open(img_path).convert("RGB")

    # Store original image for later
    orig_img_display = np.array(img)

    # --- Resize with aspect ratio preservation ---
    # Find the largest dimension and resize it to 'size'
    img.thumbnail((size, size), Image.BICUBIC)

    # Create a new 256x256 image with a black background
    new_img = Image.new("RGB", (size, size), (0, 0, 0))

    # Paste the resized image into the center
    left = (size - img.width) // 2
    top = (size - img.height) // 2
    new_img.paste(img, (left, top))
    # --- End of resize logic ---

    rgb = np.array(new_img) # This is now the 256x256 padded image
    lab = color.rgb2lab(rgb).astype("float32")
    L = lab[..., 0]
    ab = lab[..., 1:]

    # Return the padded L channel AND the original image
    return L, ab, orig_img_display

# ---------------- 5. LOAD MODELS ----------------
# --- Load ECCV16 ---
model_e_pt = ECCVGenerator().to(DEVICE)
ckpt_e = torch.load("/content/best_eccv16_pretrained.pt", map_location=DEVICE)
model_e_pt.load_state_dict(ckpt_e["model_state_dict"] if "model_state_dict" in ckpt_e else ckpt_e)
print("Loaded pretrained ECCV16 model")

# --- Load SIGGRAPH17 ---
model_s_pt = SIGGRAPHGenerator().to(DEVICE)
ckpt_s = torch.load("/content/best_sig17_pretrained.pt", map_location=DEVICE)
model_s_pt.load_state_dict(ckpt_s["model_state_dict"] if "model_state_dict" in ckpt_s else ckpt_s)
print("Loaded pretrained SIGGRAPH17 model")

# ---------------- 6. ENSEMBLE FUNCTION (FIXED) ----------------
def colorize_image_ensemble(model1, model2, img_path, size=256, device=DEVICE):
    # 1. Load image: L is 256x256 padded, orig_img is original HxW
    L, _, orig_img = load_image(img_path, size=size)
    orig_h, orig_w = orig_img.shape[:2]

    L_t = torch.from_numpy(L).unsqueeze(0).unsqueeze(0).to(device) # Shape: [1, 1, 256, 256]

    model1.eval()
    model2.eval()

    with torch.no_grad():
        pred_ab_1 = model1(L_t)
        pred_ab_2 = model2(L_t)
        pred_ab = 0.4 * pred_ab_1 + 0.6 * pred_ab_2

        # 2. Combine with padded L tensor
        out_lab_padded = torch.cat((L_t, pred_ab), dim=1)[0].cpu().numpy().transpose(1,2,0)
        out_rgb_padded = np.clip(color.lab2rgb(out_lab_padded), 0, 1) # This is 256x256

    # 3. Find dimensions to crop out padding
    # We need to know how big the image was *before* padding
    thumb = Image.open(img_path).convert("RGB")
    thumb.thumbnail((size, size), Image.BICUBIC)
    thumb_w, thumb_h = thumb.width, thumb.height

    left = (size - thumb_w) // 2
    top = (size - thumb_h) // 2
    right = left + thumb_w
    bottom = top + thumb_h

    # 4. Crop the 256x256 output to remove padding
    out_rgb_cropped = out_rgb_padded[top:bottom, left:right]

    # 5. Resize this cropped image to the *original* image size
    # Convert 0-1 float to 0-255 uint8 for cv2
    out_rgb_uint8 = (out_rgb_cropped * 255).astype(np.uint8)

    # Use cv2.resize to stretch back to original dimensions
    final_rgb = cv2.resize(out_rgb_uint8, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)

    return final_rgb, orig_img

# ---------------- 7. RUN INFERENCE ----------------
img_path = "/content/000000330369-copy.jpg" # Make sure this file exists
colored, orig = colorize_image_ensemble(model_e_pt, model_s_pt, img_path, size=256)

# Save output
out_path = "/content/output_colored_ensemble_10.png"
Image.fromarray(colored).save(out_path)
print(f"Saved ensemble colorized image at {out_path}")

# --- Display side by side ---
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
# 'orig' is now the original, full-sized image
plt.imshow(color.rgb2gray(orig), cmap='gray')
plt.title("Original (Grayscale Input)")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(colored)
plt.title("Ensemble Colorized Output (ECCV16 + SIGGRAPH17)")
plt.axis("off")

plt.show()